# Token Bank v2: Adaptive Runtime Budget Governance
## Real-World Experimental Notebook with Public Datasets

**Improvements over v1:**
1. **BudgetAdapter** — Wraps pre-execution optimizers (ZEBRA/BAMAS) as initial conditions
2. **PredictiveRebalancing** — ARIMA forecast-driven rebalancing (not threshold-based)
3. **BudgetBorrowing** — Inter-agent credit market with formal safety guarantees
4. **OnlinePriorityLearning (OPL)** — Thompson Sampling bandit for adaptive priorities
5. **KVCacheAwareQuota** — Memory-aware budgeting triggering context summarization

> **Datasets used:** GAIA (huggingface), SWE-bench-lite (github), Synthetic SEC Filings
> **API:** OpenAI GPT-4o / GPT-4o-mini (set `MOCK_MODE=True` for zero-cost debugging)


In [ ]:
# Install dependencies (run once)
!pip install openai tiktoken numpy pandas matplotlib seaborn scipy pyyaml datasets -q
!pip install statsmodels -q  # for ARIMA forecasting
print('Dependencies installed.')

In [ ]:
import os
import sys
import json
import time
import random
import re
import math
import ast
import subprocess
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple, Any
from collections import defaultdict, deque
from datetime import datetime
import copy
import threading
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import beta as beta_dist
import tiktoken

# Try to import statsmodels for ARIMA; fallback to simple EWMA if unavailable
try:
    from statsmodels.tsa.arima.model import ARIMA
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False
    print('statsmodels not available; using EWMA fallback for forecasting')

# ==================== GLOBAL CONFIG ====================

# API Configuration (OpenAI-compatible endpoint)
# Two backends, selected by LLM_BACKEND env var:
#   vllm (default): local vLLM server serving Qwen/Qwen3.6-35B-A3B
#     start with: .venv/bin/vllm serve /root/autodl-tmp/models/Qwen3.6-35B-A3B \
#       --served-model-name Qwen/Qwen3.6-35B-A3B --port 8000
#   deepseek: DeepSeek API (previous default, higher cost)
LLM_BACKEND = os.getenv('LLM_BACKEND', 'vllm')

if LLM_BACKEND == 'deepseek':
    OPENAI_API_KEY = os.getenv('DEEPSEEK_API_KEY', 'REDACTED-SEE-ENV-DEEPSEEK_API_KEY')
    OPENAI_BASE_URL = os.getenv('OPENAI_BASE_URL', 'https://api.deepseek.com')
    PRIMARY_MODEL = os.getenv('DEEPSEEK_MODEL', 'deepseek-v4-pro')
    THROTTLE_MODEL = 'deepseek-v4-flash'   # cheaper tier, used when throttling
    SUMMARIZE_MODEL = 'deepseek-v4-flash'  # for KV-cache summarization
    # Pricing (USD per 1M tokens, DeepSeek V4 list prices, cache-miss input)
    PRICING = {
        'deepseek-v4-pro': {'input': 0.435, 'output': 0.87},
        'deepseek-v4-flash': {'input': 0.14, 'output': 0.28},
    }
    LLM_EXTRA_BODY = {}
else:
    OPENAI_API_KEY = os.getenv('VLLM_API_KEY', 'EMPTY')
    OPENAI_BASE_URL = os.getenv('OPENAI_BASE_URL', 'http://127.0.0.1:8000/v1')
    PRIMARY_MODEL = os.getenv('VLLM_MODEL', 'Qwen/Qwen3.6-35B-A3B')
    THROTTLE_MODEL = PRIMARY_MODEL    # single local model: no cheaper tier
    SUMMARIZE_MODEL = PRIMARY_MODEL   # judge/summarization reuse the same model
    # Nominal local pricing for CPS continuity (GPU rental amortized per token,
    # NOT an API bill). Relative cost comparisons across configs stay valid.
    PRICING = {
        PRIMARY_MODEL: {'input': 0.03, 'output': 0.03},
    }
    # Qwen3.6 / Laguna are hybrid-thinking models; thinking traces would eat the
    # small max_tokens budgets used here (same failure mode as DeepSeek V4
    # thinking). Gemma doesn't accept this chat-template kwarg.
    # Mistral Small 4 uses reasoning_effort instead (official per-request switch).
    # REASONING_EFFORT env: 'none' (default, protocol parity) or 'high' (reasoning
    # variant — pair with MAX_TOKENS >= 4096 and larger budgets).
    LLM_EXTRA_BODY = {}
    if 'istral' in PRIMARY_MODEL:
        LLM_EXTRA_BODY = {'reasoning_effort': os.getenv('REASONING_EFFORT', 'none')}
    elif 'qwen' in PRIMARY_MODEL.lower() or 'laguna' in PRIMARY_MODEL.lower():
        LLM_EXTRA_BODY = {'chat_template_kwargs': {'enable_thinking': False}}

# Per-call output cap. Default 1024 (fast-answer protocol). Reasoning variants
# need more headroom for thinking traces (dispatch estimates scale with this).
MAX_TOKENS = int(os.getenv('MAX_TOKENS', '1024'))

# Experiment Mode
MOCK_MODE = False          # True: simulate LLM responses (zero cost)
VERBOSE = True
SAVE_TRAJECTORIES = True

# Model Configuration (set in the backend blocks above)
print(f'LLM Backend: {LLM_BACKEND} | Base URL: {OPENAI_BASE_URL} | Model: {PRIMARY_MODEL}')

# Token encoder for budget accounting: prefer the served model's own tokenizer
# (accurate per-token accounting); fall back to tiktoken gpt-4o when the local
# tokenizer is unavailable (e.g. running against a remote API on a dev laptop).
class _HFEncoder:
    def __init__(self, path):
        from transformers import AutoTokenizer
        self._tok = AutoTokenizer.from_pretrained(path)

    def encode(self, text):
        return self._tok.encode(text)

TOKENIZER_PATH = os.getenv(
    'TOKENIZER_PATH', f'/root/autodl-tmp/models/{PRIMARY_MODEL.split("/")[-1]}')
try:
    ENCODER = _HFEncoder(TOKENIZER_PATH) if LLM_BACKEND != 'deepseek' else None
    if ENCODER is None:
        raise ValueError('deepseek backend: use tiktoken')
    print(f'Tokenizer: {TOKENIZER_PATH}')
except Exception as e:
    print(f'Local tokenizer unavailable ({e}); falling back to tiktoken gpt-4o')
    ENCODER = tiktoken.encoding_for_model('gpt-4o')

# Mock-mode simulated per-call token sizes (mean, std) per role.
# Sized so that a single call consumes a meaningful fraction of the per-agent
# quota, so borrowing / truncation / throttling actually get exercised.
MOCK_TOKEN_PROFILES = {
    'Planner': (8000, 2000),
    'Executor': (15000, 4000),
    'Reviewer': (5000, 1500),
    'Debugger': (12000, 3500),
}

# Random seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Experiment budgets per benchmark (tokens)
BUDGETS = {
    'GAIA': 100000,
    'SWE': 150000,
    'FINANCIAL': 80000,
}

print(f'Mock Mode: {MOCK_MODE}')
print(f'Primary Model: {PRIMARY_MODEL}')
print(f'Has statsmodels ARIMA: {HAS_STATSMODELS}')

## Dataset Loading

We use three publicly available / reproducible datasets:

1. **GAIA** (huggingface `gaia-benchmark/GAIA`): General AI Assistant benchmark with multi-step reasoning tasks. We use validation split, levels 1–2.
2. **SWE-bench Lite** (Princeton-NLP GitHub): 300 Python bug-fix tasks. We load from the official JSON and use a 50-task subset.
3. **Financial Document Extraction** (synthetic SEC 10-K excerpts): Publicly available sample filings from Apple, Tesla, Microsoft. We construct structured extraction tasks.

> **Note:** If huggingface datasets are unavailable, the code falls back to cached local JSON files.

In [ ]:
# ==================== DATASET: GAIA ====================
def _extract_attachment_text(path: str, max_chars: int = 3000) -> str:
    """Extract text from a GAIA attachment (pdf/docx/xlsx/ plain text)."""
    ext = os.path.splitext(path)[1].lower()
    try:
        if ext == '.pdf':
            import pypdf
            reader = pypdf.PdfReader(path)
            text = '\n'.join((p.extract_text() or '') for p in reader.pages[:5])
        elif ext == '.docx':
            import docx
            text = '\n'.join(par.text for par in docx.Document(path).paragraphs)
        elif ext == '.xlsx':
            import openpyxl
            wb = openpyxl.load_workbook(path, read_only=True)
            rows = []
            for ws in wb.worksheets:
                for row in ws.iter_rows(values_only=True):
                    rows.append(' '.join('' if c is None else str(c) for c in row))
            text = '\n'.join(rows)
        elif ext == '.pptx':
            import pptx
            prs = pptx.Presentation(path)
            text = '\n'.join(shape.text for slide in prs.slides for shape in slide.shapes
                             if shape.has_text_frame)
        elif ext == '.zip':
            import zipfile, tempfile
            parts = []
            with zipfile.ZipFile(path) as zf, tempfile.TemporaryDirectory() as tmp:
                zf.extractall(tmp)
                for name in zf.namelist()[:10]:
                    p = os.path.join(tmp, name)
                    if os.path.isfile(p) and os.path.getsize(p) < 200_000:
                        try:
                            with open(p, 'r', encoding='utf-8', errors='ignore') as f:
                                parts.append(f'--- {name} ---\n{f.read()[:1500]}')
                        except Exception:
                            pass
            text = '\n'.join(parts)
        else:
            with open(path, 'r', encoding='utf-8', errors='ignore') as f:
                text = f.read()
    except Exception as e:
        return f'[attachment could not be parsed: {e}]'
    return text[:max_chars]


def _load_gaia_attachment(file_name: str) -> str:
    """Download a GAIA attachment from the (gated) HF dataset repo and extract text."""
    from huggingface_hub import hf_hub_download
    path = hf_hub_download(repo_id='gaia-benchmark/GAIA', repo_type='dataset',
                           filename=f'2023/validation/{file_name}')
    return _extract_attachment_text(path)


def load_gaia_dataset(n_tasks: int = 50, level_filter: List[int] = [1, 2]) -> List[Dict]:
    """Load GAIA benchmark from huggingface or fallback JSON."""
    # Binary media (image/audio) is unanswerable for a text-only model;
    # those tasks are excluded from sampling and counted here.
    MEDIA_EXTS = ('.png', '.jpg', '.jpeg', '.mp3', '.wav')
    try:
        from datasets import load_dataset
        # Gated dataset: requires HF_TOKEN env var with accepted terms
        ds = load_dataset('gaia-benchmark/GAIA', '2023_all', split='validation')
        tasks = []
        n_media_skipped = 0
        for item in ds:
            lvl = item.get('Level', '1')
            try:
                lvl_int = int(lvl)
            except:
                lvl_int = 1
            if lvl_int in level_filter and item.get('Question') and item.get('Final answer'):
                question = item['Question']
                file_name = item.get('file_name') or ''
                if file_name and file_name.lower().endswith(MEDIA_EXTS):
                    n_media_skipped += 1
                    continue
                if file_name:
                    try:
                        attachment = _load_gaia_attachment(file_name)
                        question += f'\n\n[Attached file: {file_name}]\n{attachment}'
                    except Exception as e:
                        print(f'  attachment {file_name} unavailable: {e}')
                tasks.append({
                    'id': f"gaia_{item.get('task_id', len(tasks))}",
                    'question': question,
                    'answer': str(item['Final answer']),
                    'level': lvl_int,
                    'has_attachment': bool(file_name),
                    'expected_agents': ['Planner', 'Executor', 'Reviewer']
                })
        if n_media_skipped:
            print(f'GAIA: skipped {n_media_skipped} image/audio tasks (text-only model)')
        random.shuffle(tasks)
        return tasks[:n_tasks]
    except Exception as e:
        print(f'GAIA huggingface load failed: {e}')
        print('Falling back to synthetic GAIA-like tasks...')
        return load_gaia_fallback(n_tasks)

def load_gaia_fallback(n_tasks: int = 50) -> List[Dict]:
    """Synthetic GAIA-like tasks when huggingface is unavailable."""
    templates = [
        {'q': 'What is the population of {city} (capital of {country})?', 'a': '{pop}'},
        {'q': 'Who composed the score for the {year} film {film}?', 'a': '{composer}'},
        {'q': 'Calculate compound interest on ${p} at {r}% for {t} years, compounded monthly.', 'a': '{result}'},
        {'q': 'Which NASA mission first landed a rover on Mars? Year? Rover name?', 'a': 'Mars Pathfinder; 1997; Sojourner'},
        {'q': 'Sum of all prime numbers between {a} and {b}.', 'a': '{result}'},
    ]
    fillers = [
        {'city': 'Paris', 'country': 'France', 'pop': '2,161,000'},
        {'year': '2023', 'film': 'Oppenheimer', 'composer': 'Ludwig Göransson'},
        {'p': '10000', 'r': '5', 't': '3', 'result': '1614.72'},
        {'a': '100', 'b': '200', 'result': '3167'},
    ]
    # template index -> matching filler index (template 3 needs no filler)
    filler_for_template = {0: 0, 1: 1, 2: 2, 3: 0, 4: 3}
    tasks = []
    for i in range(n_tasks):
        ti = i % len(templates)
        t = templates[ti]
        f = fillers[filler_for_template[ti]]
        q = t['q'].format(**f)
        a = t['a'].format(**f)
        tasks.append({'id': f'gaia_fallback_{i}', 'question': q, 'answer': a, 'level': 1,
                      'expected_agents': ['Planner', 'Executor', 'Reviewer']})
    return tasks

# ==================== DATASET: SWE-bench Lite ====================
def _swe_context_from_gold_patch(patch: str) -> str:
    """Reconstruct the buggy code region from the gold patch's minus/context
    lines — what an agent with repo access would read. Added lines (the fix
    itself) are excluded so the answer is not leaked."""
    lines = []
    for ln in patch.split('\n'):
        if ln.startswith(('+++', '---', '@@')):
            continue
        if ln.startswith('-') or ln.startswith(' '):
            lines.append(ln[1:])
    return '\n'.join(lines)


def load_swebench_lite(n_tasks: int = 30) -> List[Dict]:
    """Load SWE-bench-lite from HuggingFace or fallback."""
    try:
        from datasets import load_dataset
        ds = load_dataset('princeton-nlp/SWE-bench_Lite', split='test')
        tasks = []
        for item in list(ds)[:n_tasks]:
            tasks.append({
                'id': f"swe_{item.get('instance_id', len(tasks))}",
                'issue': item.get('problem_statement', 'Fix the bug'),
                'repo': item.get('repo', ''),
                'code_before': _swe_context_from_gold_patch(item.get('patch', '')),
                'gold_patch': item.get('patch', ''),
                'expected_agents': ['Planner', 'Executor', 'Debugger', 'Reviewer']
            })
        return tasks
    except Exception as e:
        print(f'SWE-bench load failed: {e}')
        return load_swe_fallback(n_tasks)

def load_swe_fallback(n_tasks: int = 30) -> List[Dict]:
    """Synthetic SWE-like tasks."""
    tasks = [
        {'id': 'swe_001', 'issue': 'Fix off-by-one error in list slicing when end index equals list length.',
         'code_before': 'def slice_list(lst, start, end):\n    return lst[start:end+1]',
         'code_after': 'def slice_list(lst, start, end):\n    return lst[start:end]',
         'test': 'assert slice_list([1,2,3], 0, 3) == [1,2,3]'},
        {'id': 'swe_002', 'issue': 'Handle KeyError when dict key might not exist; return default value None.',
         'code_before': 'def get_value(d, key):\n    return d[key]',
         'code_after': 'def get_value(d, key):\n    return d.get(key)',
         'test': 'assert get_value({}, "x") is None'},
        {'id': 'swe_003', 'issue': 'Fix recursive Fibonacci to use memoization to avoid exponential time.',
         'code_before': 'def fib(n):\n    if n <= 1: return n\n    return fib(n-1) + fib(n-2)',
         'code_after': 'def fib(n, memo={}):\n    if n in memo: return memo[n]\n    if n <= 1: return n\n    memo[n] = fib(n-1, memo) + fib(n-2, memo)\n    return memo[n]',
         'test': 'assert fib(30) == 832040'},
    ]
    result = []
    for i in range(n_tasks):
        t = copy.deepcopy(tasks[i % len(tasks)])
        t['id'] = f'swe_fallback_{i}'
        result.append(t)
    return result

# ==================== DATASET: Financial (Synthetic SEC 10-K) ====================
def load_financial_dataset(n_tasks: int = 20) -> List[Dict]:
    """Synthetic financial extraction tasks based on public SEC filing excerpts."""
    docs = [
        {
            'id': 'fin_001',
            'company': 'Apple Inc.',
            'document': """Apple Inc. 10-K Filing (Excerpt)
CEO: Timothy D. Cook
Headquarters: Cupertino, California
Fiscal Year 2024 Revenue: $391.0 billion
Net Income: $101.9 billion
Total Employees: 161,000
Board Members: Arthur Levinson, James Bell, Al Gore, Andrea Jung
Audit Firm: Ernst & Young LLP
Stock Ticker: AAPL
Fiscal Year End: Last Saturday of September""",
            'fields': {
                'ceo': 'Timothy D. Cook', 'headquarters': 'Cupertino, California',
                'revenue': '$391.0 billion', 'net_income': '$101.9 billion',
                'employees': '161,000', 'audit_firm': 'Ernst & Young LLP',
                'ticker': 'AAPL', 'fy_end': 'Last Saturday of September'
            }
        },
        {
            'id': 'fin_002',
            'company': 'Tesla Inc.',
            'document': """Tesla Inc. 10-K Filing (Excerpt)
CEO: Elon Musk
Incorporated: Delaware
Principal Executive Offices: Austin, Texas
2024 Revenue: $97.7 billion
Gross Profit: $17.9 billion
Cash and Cash Equivalents: $14.7 billion
Total Debt: $9.6 billion
Independent Auditor: PricewaterhouseCoopers LLP
Stock Ticker: TSLA
Employees: 125,000""",
            'fields': {
                'ceo': 'Elon Musk', 'incorporated': 'Delaware',
                'headquarters': 'Austin, Texas', 'revenue': '$97.7 billion',
                'gross_profit': '$17.9 billion', 'cash': '$14.7 billion',
                'debt': '$9.6 billion', 'auditor': 'PricewaterhouseCoopers LLP',
                'ticker': 'TSLA', 'employees': '125,000'
            }
        },
        {
            'id': 'fin_003',
            'company': 'Microsoft Corp.',
            'document': """Microsoft Corporation 10-K Filing (Excerpt)
CEO: Satya Nadella
Headquarters: Redmond, Washington
Fiscal Year 2024 Revenue: $245.1 billion
Operating Income: $109.4 billion
Net Income: $88.1 billion
Total Employees: 228,000
Independent Auditor: Deloitte & Touche LLP
Stock Ticker: MSFT
Fiscal Year End: June 30""",
            'fields': {
                'ceo': 'Satya Nadella', 'headquarters': 'Redmond, Washington',
                'revenue': '$245.1 billion', 'operating_income': '$109.4 billion',
                'net_income': '$88.1 billion', 'employees': '228,000',
                'auditor': 'Deloitte & Touche LLP', 'ticker': 'MSFT',
                'fy_end': 'June 30'
            }
        },
    ]
    result = []
    for i in range(n_tasks):
        d = copy.deepcopy(docs[i % len(docs)])
        d['id'] = f"fin_{i:03d}"
        d['expected_agents'] = ['Planner', 'Executor', 'Reviewer', 'Debugger']
        result.append(d)
    return result

# ==================== LOAD ALL ====================
GAIA_TASKS = load_gaia_dataset(n_tasks=100)
SWE_TASKS = load_swebench_lite(n_tasks=30)
FINANCIAL_TASKS = load_financial_dataset(n_tasks=20)

print(f'Loaded datasets:')
print(f'  GAIA: {len(GAIA_TASKS)} tasks')
print(f'  SWE-bench Lite: {len(SWE_TASKS)} tasks')
print(f'  Financial: {len(FINANCIAL_TASKS)} tasks')
print(f'  Total: {len(GAIA_TASKS) + len(SWE_TASKS) + len(FINANCIAL_TASKS)} tasks')

# Optional smoke-test subset: SMOKE_N env var truncates each task set and
# skips the ablation / sensitivity sweeps (for quick real-API checks)
SMOKE_N = int(os.getenv('SMOKE_N', '0'))
if SMOKE_N:
    GAIA_TASKS = GAIA_TASKS[:SMOKE_N]
    SWE_TASKS = SWE_TASKS[:SMOKE_N]
    FINANCIAL_TASKS = FINANCIAL_TASKS[:SMOKE_N]
    print(f'SMOKE_N={SMOKE_N}: task sets truncated, ablation/sensitivity skipped')

# Smoke runs write artifacts to *_smoke.* so they don't clobber full-run outputs
OUT_SUFFIX = '_smoke' if SMOKE_N else ''

## TokenBank v2 Core Implementation

Components:
- **BudgetAdapter**: Imports external pre-execution plans (ZEBRA/BAMAS/manual)
- **PredictiveRebalancer**: ARIMA-based forecast + opportunity cost rebalancing
- **BudgetBorrowing**: Inter-agent credit market with formal safety guarantees
- **OnlinePriorityLearner**: Thompson Sampling bandit for adaptive priorities
- **KVCacheAwareQuota**: Memory-aware effective token tracking

In [ ]:
@dataclass
class AgentAccountV2:
    agent_id: str
    role: str
    quota: int = 0
    used: int = 0
    priority: int = 2
    borrowed: int = 0       # credit line utilized
    lent: int = 0           # credit extended to others
    history: List[Dict] = field(default_factory=list)
    consumption_log: deque = field(default_factory=lambda: deque(maxlen=100))
    kv_cache_tokens: int = 0

    @property
    def remaining(self) -> int:
        return max(0, self.quota + self.borrowed - self.used)

    @property
    def utilization(self) -> float:
        denom = self.quota + self.borrowed
        return self.used / denom if denom > 0 else 0.0

    @property
    def effective_tokens(self) -> int:
        """KV-cache aware: prompt + 0.3 * kv_cache"""
        return self.used + int(0.3 * self.kv_cache_tokens)

    @property
    def total_cost(self) -> float:
        return sum(h.get('cost_usd', 0) for h in self.history)


class BudgetAdapter:
    """Wraps pre-execution optimizer outputs into initial quotas."""
    def __init__(self, external_plan: Optional[Dict] = None):
        self.plan = external_plan or {}

    def get_initial_quotas(self, total_budget: int, n_agents: int,
                           role_map: Dict[str, str]) -> Dict[str, int]:
        if not self.plan:
            return {aid: total_budget // n_agents for aid in role_map}
        quotas = {}
        for aid, role in role_map.items():
            q = self.plan.get(aid, self.plan.get(role, total_budget // n_agents))
            quotas[aid] = int(q)
        return quotas

    def compute_deviation(self, planned: int, actual: int) -> float:
        return abs(actual - planned) / max(planned, 1)


class PredictiveRebalancer:
    """ARIMA/EWMA forecast + opportunity cost-driven rebalancing."""
    def __init__(self, horizon: int = 2, min_history: int = 1):
        self.horizon = horizon
        self.min_history = min_history

    def forecast(self, history: deque) -> Tuple[float, float]:
        """Returns (expected_consumption_over_horizon, trend). Agents with no
        history forecast 0, i.e. their unused quota counts as surplus."""
        if len(history) < self.min_history:
            return 0.0, 0.0
        series = list(history)
        if HAS_STATSMODELS and len(series) >= 5:
            try:
                model = ARIMA(series, order=(1,1,1))
                fitted = model.fit()
                fcast = fitted.forecast(steps=self.horizon)
                expected = sum(max(0, x) for x in fcast)
                trend = float(fcast[-1] - fcast[0]) if self.horizon > 1 else 0.0
                return expected, trend
            except:
                pass
        # EWMA fallback
        alpha = 0.3
        ewma = series[0]
        for x in series[1:]:
            ewma = alpha * x + (1 - alpha) * ewma
        expected = ewma * self.horizon
        trend = series[-1] - series[0]
        return expected, trend

    def should_rebalance(self, agents: List[AgentAccountV2],
                         reserve: int, threshold: float = 0.8) -> Tuple[bool, Dict]:
        """Opportunity cost-based rebalancing decision.

        threshold (theta) is the required headroom: an agent is predicted
        deficient when its expected near-term consumption exceeds `threshold`
        of its remaining quota. Lower theta -> triggers earlier."""
        forecasts = {}
        for a in agents:
            exp, trend = self.forecast(a.consumption_log)
            forecasts[a.agent_id] = {'expected': exp, 'trend': trend,
                                      'predicted_surplus': a.remaining - exp}

        deficit_agents = [a for a in agents
                          if forecasts[a.agent_id]['expected'] > threshold * a.remaining]
        surplus_agents = [a for a in agents if forecasts[a.agent_id]['predicted_surplus'] > 0]

        if not deficit_agents or not surplus_agents:
            return False, {}

        cost_miss = sum(a.priority * forecasts[a.agent_id]['expected']
                        for a in deficit_agents)
        reclaimable = sum((a.quota - a.used) * 0.3 for a in surplus_agents if a.utilization < 0.5)
        cost_rebal = reclaimable * 0.05

        return cost_miss > cost_rebal, forecasts


class CreditScorer:
    """Bank-style credit scoring: tracks whether borrowed credit converts to
    task success ('repayment') per role, and sizes/prices credit accordingly.
    Thread-safe so it can be shared across concurrent task banks."""
    def __init__(self, base_rate: float = 0.5, base_interest: float = 0.15):
        self.base_rate = base_rate
        self.base_interest = base_interest
        self.history: Dict[str, Dict[str, float]] = {}
        self._lock = threading.RLock()

    def score(self, role: str) -> float:
        with self._lock:
            h = self.history.get(role, {'alpha': 1.0, 'beta': 1.0})
            return h['alpha'] / (h['alpha'] + h['beta'])  # posterior mean, starts at 0.5

    def credit_rate_for(self, role: str) -> float:
        """score 0 -> 0.5x base_rate, score 1 -> 1.5x base_rate."""
        return self.base_rate * (0.5 + self.score(role))

    def interest_for(self, role: str) -> float:
        """Risk pricing: score 0 -> 2x base interest, score 1 -> 0.5x."""
        return self.base_interest * (1.5 - self.score(role))

    def record(self, role: str, repaid: bool) -> None:
        with self._lock:
            h = self.history.setdefault(role, {'alpha': 1.0, 'beta': 1.0})
            h['alpha' if repaid else 'beta'] += 1

    def summary(self) -> Dict[str, float]:
        with self._lock:
            return {r: round(self.score(r), 3) for r in self.history}


class BudgetBorrowing:
    """Inter-agent credit market with safety guarantees (Theorem 2)."""
    def __init__(self, credit_rate: float = 0.10, interest_rate: float = 0.15,
                 scorer: 'CreditScorer' = None, rogue_cutoff: float = 0.10):
        self.credit_rate = credit_rate
        self.interest_rate = interest_rate
        self.scorer = scorer
        # Option C: credit score is used to size/pricing credit lines (shallow)
        # and to hard-cutoff confirmed rogue roles, but does not gate normal
        # peer-to-peer lending decisions.
        self.rogue_cutoff = rogue_cutoff
        self.loans: List[Dict] = []

    def max_credit_line(self, agent: AgentAccountV2) -> int:
        rate = self.scorer.credit_rate_for(agent.role) if self.scorer else self.credit_rate
        return int(agent.quota * rate)

    def can_borrow(self, borrower: AgentAccountV2, amount: int,
                   lender: AgentAccountV2, reserve: int) -> bool:
        """Theorem 2 safety check + rogue-only cutoff."""
        if amount <= 0:
            return False
        if self.scorer:
            score = self.scorer.score(borrower.role)
            # Confirmed rogues (very low credit) are cut off entirely.
            if score < self.rogue_cutoff:
                return False
        if borrower.borrowed + amount > self.max_credit_line(borrower):
            return False
        if lender.priority > borrower.priority:
            return False
        if lender.remaining < amount:
            return False
        if reserve <= 0:
            return False
        return True

    def execute_loan(self, borrower: AgentAccountV2, lender: AgentAccountV2,
                     amount: int) -> Dict:
        rate = self.scorer.interest_for(borrower.role) if self.scorer else self.interest_rate
        interest = int(amount * rate)
        borrower.borrowed += amount
        lender.lent += amount + interest
        lender.quota -= amount
        self.loans.append({
            'borrower': borrower.agent_id, 'lender': lender.agent_id,
            'amount': amount, 'interest': interest, 'timestamp': time.time()
        })
        return {'success': True, 'amount': amount, 'interest': interest}


class OnlinePriorityLearner:
    """Thompson Sampling for adaptive role priorities."""
    def __init__(self, roles: List[str], cold_start_tasks: int = 10):
        self.roles = roles
        self.posteriors = {r: {'alpha': 1.0, 'beta': 1.0} for r in roles}
        self.cold_start = cold_start_tasks
        self.task_count = 0
        self.manual_priorities = {'Planner': 3, 'Executor': 2, 'Reviewer': 2, 'Debugger': 1}

    def get_priorities(self) -> Dict[str, int]:
        if self.task_count < self.cold_start:
            return self.manual_priorities.copy()
        priorities = {}
        for r in self.roles:
            p = beta_dist.rvs(self.posteriors[r]['alpha'], self.posteriors[r]['beta'])
            priorities[r] = max(1, min(3, round(1 + 2 * p)))
        return priorities

    def update(self, role_success: Dict[str, bool], role_blocked: Dict[str, bool]) -> None:
        self.task_count += 1
        for role in self.roles:
            if role_success.get(role, False):
                self.posteriors[role]['alpha'] += 1
            elif role_blocked.get(role, False):
                self.posteriors[role]['beta'] += 1

    def get_regret_bound(self, T: int) -> float:
        K = len(self.roles)
        return math.sqrt(K * T * math.log(T + 1))


class KVCacheTracker:
    """Tracks KV-cache accumulation and triggers summarization."""
    def __init__(self, compression_factor: float = 0.3, trigger_ratio: float = 0.85):
        self.compression_factor = compression_factor
        self.trigger_ratio = trigger_ratio
        self.summarization_count = 0

    def effective_tokens(self, prompt_tokens: int, kv_cache: int) -> int:
        return prompt_tokens + int(self.compression_factor * kv_cache)

    def should_summarize(self, agent: AgentAccountV2) -> bool:
        eff = agent.effective_tokens
        return eff / max(agent.quota, 1) > self.trigger_ratio

    def summarize_context(self, messages: List[Dict], llm_client) -> Tuple[List[Dict], int, int]:
        self.summarization_count += 1
        summary_msg = {
            'role': 'system',
            'content': '[Context summarized due to KV-cache pressure. Prior conversation condensed.]'
        }
        kept = messages[-2:] if len(messages) > 2 else messages
        new_messages = [summary_msg] + kept
        tokens_in = sum(len(ENCODER.encode(m.get('content', ''))) for m in new_messages)
        return new_messages, tokens_in, 0

print('All v2 components defined.')

In [ ]:
class TokenBankV2:
    """Adaptive Runtime Budget Governance Middleware."""
    def __init__(self, total_budget: int, policy='hard', external_plan=None,
                 gamma=0.8, rebalance_threshold=0.8, enable_predictive=True,
                 enable_borrowing=True, enable_opl=True, enable_kv=True,
                 min_quota=5000, anomaly_sigma=3.0, shared_opl=None,
                 shared_credit_scorer=None, rogue_cutoff=0.10,
                 credit_rate=0.5, enable_pacing=False):
        self.total_budget = total_budget
        self.reserve = total_budget
        self.policy = policy
        self.gamma = gamma
        self.theta = rebalance_threshold
        self.q_min = min_quota
        self.anomaly_sigma = anomaly_sigma
        self.adapter = BudgetAdapter(external_plan)
        self.predictor = PredictiveRebalancer(horizon=2) if enable_predictive else None
        self.borrowing = BudgetBorrowing(
            credit_rate=credit_rate, scorer=shared_credit_scorer,
            rogue_cutoff=rogue_cutoff
        ) if enable_borrowing else None
        self.credit_scorer = shared_credit_scorer
        self.rogue_cutoff = rogue_cutoff
        # shared_opl: an OnlinePriorityLearner shared across tasks so priority
        # learning accumulates beyond a single task (else cold-start never ends)
        self.opl = shared_opl
        self.enable_opl = enable_opl
        self.kv_tracker = KVCacheTracker() if enable_kv else None
        # Budget pacing (soft segmentation): the first 50% of the budget is
        # available upfront; the back 50% is split into segments that are
        # released only when (a) >=80% of the released pool is consumed and
        # (b) a Reviewer checkpoint has completed since the last release.
        # Global ceiling independent of per-agent quota/borrowing.
        self.pacing = enable_pacing
        self.pacing_k = 3
        self.released = int(total_budget * 0.5)
        # ceil so the final release tops up to exactly total_budget
        self.segment_size = -(-(total_budget - self.released) // self.pacing_k)
        self._reviews_since_release = 0
        self.pacing_releases = 0
        self.ledger = {}
        self.rebalance_count = 0
        self.borrow_count = 0
        self.anomaly_log = []
        self.trajectory = []
        self._velocity_window = defaultdict(list)

    def total_used(self):
        return sum(a.used for a in self.ledger.values())

    def pacing_release_check(self, action):
        """Called after every successful consume(). Releases the next pacing
        segment when the released pool is >=80% consumed AND a Reviewer
        checkpoint happened since the last release."""
        if not self.pacing or self.released >= self.total_budget:
            return
        if action == 'review':
            self._reviews_since_release += 1
        if (self.total_used() >= 0.8 * self.released
                and self._reviews_since_release >= 1):
            self.released = min(self.total_budget, self.released + self.segment_size)
            self.pacing_releases += 1
            self._reviews_since_release = 0
            self._log_event('PACING_RELEASE', {'released': self.released})

    def register_agents(self, role_map, priorities=None):
        for aid, role in role_map.items():
            p = priorities.get(role, 2) if priorities else 2
            self.ledger[aid] = AgentAccountV2(agent_id=aid, role=role, priority=p)
        if self.enable_opl and self.opl is None:
            self.opl = OnlinePriorityLearner(list(set(role_map.values())))

    def allocate(self, priority_weight=True):
        n = len(self.ledger)
        if n == 0:
            return
        role_map = {aid: acc.role for aid, acc in self.ledger.items()}
        plan_quotas = self.adapter.get_initial_quotas(self.total_budget, n, role_map)
        if self.opl and self.opl.task_count >= self.opl.cold_start:
            learned_p = self.opl.get_priorities()
        else:
            learned_p = {acc.role: acc.priority for acc in self.ledger.values()}

        # Option C: initial allocation uses role priority only, not credit score.
        total_p = sum(learned_p.get(acc.role, 2) for acc in self.ledger.values())
        available = self.total_budget - n * self.q_min
        runtime_quotas = {}
        for aid, acc in self.ledger.items():
            p = learned_p.get(acc.role, 2)
            runtime_quotas[aid] = max(self.q_min, int(p * available / total_p))
        for aid in self.ledger:
            q_plan = plan_quotas.get(aid, self.total_budget // n)
            q_runtime = runtime_quotas.get(aid, self.total_budget // n)
            self.ledger[aid].quota = int(self.gamma * q_plan + (1 - self.gamma) * q_runtime)
        self._log_event('ALLOCATE', {k: self.ledger[k].quota for k in self.ledger})

    def can_dispatch(self, agent_id, estimated_tokens=0):
        if agent_id not in self.ledger:
            return False, 'Agent not registered'
        # Pacing gate: global ceiling on task-level consumption, independent of
        # per-agent quota and borrowing. Blocks until a Reviewer checkpoint
        # releases the next segment (fail-fast on runaway tasks).
        if self.pacing and self.total_used() + estimated_tokens > self.released:
            return False, f'Pacing gate: {self.total_used()}/{self.released} awaiting checkpoint'
        acc = self.ledger[agent_id]
        grace = 1.0
        if self.policy == 'grace':
            grace = {3: 1.10, 2: 1.05, 1: 1.00}.get(acc.priority, 1.0)
        effective_quota = int((acc.quota + acc.borrowed) * grace)
        if acc.used + estimated_tokens > effective_quota:
            if self.borrowing and self._try_borrow(agent_id, estimated_tokens - acc.remaining):
                return True, 'OK (after borrowing)'
            return False, f'Quota exceeded: {acc.used}/{effective_quota}'
        if estimated_tokens > self.reserve:
            return False, f'Global reserve depleted: {self.reserve}'
        return True, 'OK'

    def _try_borrow(self, borrower_id, amount):
        if not self.borrowing or amount <= 0:
            return False
        borrower = self.ledger[borrower_id]
        candidates = [a for a in self.ledger.values()
                      if a.agent_id != borrower_id and a.remaining > amount]
        # Option C: normal lending decisions are not gated by credit score.
        # Sort by priority (protect important lenders) then remaining.
        candidates.sort(key=lambda a: (a.priority, -a.remaining))
        for lender in candidates:
            if self.borrowing.can_borrow(borrower, amount, lender, self.reserve):
                result = self.borrowing.execute_loan(borrower, lender, amount)
                self.borrow_count += 1
                self._log_event('BORROW', result)
                return True
        return False

    def consume(self, agent_id, tokens_in, tokens_out, model=PRIMARY_MODEL,
                action='inference', kv_cache=0):
        if agent_id not in self.ledger:
            return {'allowed': False, 'reason': 'Agent not registered'}
        acc = self.ledger[agent_id]
        acc.kv_cache_tokens += kv_cache
        if self.kv_tracker:
            total_tokens = self.kv_tracker.effective_tokens(tokens_in, acc.kv_cache_tokens) + tokens_out
        else:
            total_tokens = tokens_in + tokens_out
        cost = self._compute_cost(tokens_in, tokens_out, model)
        if self.policy == 'throttle' and acc.utilization > 0.9:
            model = THROTTLE_MODEL
            cost = self._compute_cost(tokens_in, tokens_out, model)
            self._log_event('THROTTLE', {'agent': agent_id, 'new_model': model})
        grace = 1.0
        if self.policy == 'grace':
            grace = {3: 1.10, 2: 1.05, 1: 1.00}.get(acc.priority, 1.0)
        effective_quota = int((acc.quota + acc.borrowed) * grace)
        if acc.used + total_tokens > effective_quota:
            allowed_tokens = max(0, effective_quota - acc.used)
            if self.policy == 'hard' or allowed_tokens == 0:
                return {'allowed': False, 'actual_in': 0, 'actual_out': 0,
                        'reason': f'Quota exceeded: {acc.used}/{effective_quota}'}
            total_tokens = allowed_tokens
            tokens_out = min(tokens_out, allowed_tokens)
            tokens_in = allowed_tokens - tokens_out
        if total_tokens > self.reserve:
            return {'allowed': False, 'actual_in': 0, 'actual_out': 0,
                    'reason': f'Global reserve depleted: {self.reserve}'}
        acc.used += total_tokens
        self.reserve -= total_tokens
        acc.consumption_log.append(total_tokens)
        record = {'timestamp': datetime.now().isoformat(), 'tokens_in': tokens_in,
                  'tokens_out': tokens_out, 'kv_cache': kv_cache, 'model': model,
                  'cost_usd': cost, 'action': action}
        acc.history.append(record)
        self._log_event('CONSUME', {'agent': agent_id, 'tokens': total_tokens, 'cost': cost})
        self._check_anomaly(agent_id, total_tokens)
        return {'allowed': True, 'actual_in': tokens_in, 'actual_out': tokens_out,
                'model': model, 'cost_usd': cost, 'reason': 'OK'}

    def predictive_rebalance(self):
        if not self.predictor:
            return {'performed': False, 'reason': 'Disabled'}
        agents = list(self.ledger.values())
        should, forecasts = self.predictor.should_rebalance(agents, self.reserve, self.theta)
        if not should:
            return {'performed': False, 'reason': 'Opportunity cost too low', 'forecasts': forecasts}
        surplus = [a for a in agents if forecasts[a.agent_id]['predicted_surplus'] > 0 and a.utilization < 0.5]
        deficit = [a for a in agents if forecasts[a.agent_id]['expected'] > self.theta * a.remaining]
        if not surplus or not deficit:
            return {'performed': False, 'reason': 'No matching pairs', 'forecasts': forecasts}
        reclaimable = int(sum((a.quota - a.used) * 0.3 for a in surplus))
        if reclaimable <= 0:
            return {'performed': False, 'reason': 'No reclaimable quota'}
        # Option C: rebalancing uses role priority only, not credit score.
        deficit.sort(key=lambda a: a.priority, reverse=True)
        total_p = sum(a.priority for a in deficit) or 1
        transfers = []
        for a in deficit:
            share = int(reclaimable * (a.priority / total_p))
            if share > 0:
                a.quota += share
                transfers.append({'to': a.agent_id, 'amount': share})
        total_surplus_rem = sum(a.remaining for a in surplus) or 1
        for a in surplus:
            deduction = int(reclaimable * (a.remaining / total_surplus_rem))
            a.quota = max(self.q_min, a.quota - deduction)
        self.rebalance_count += 1
        self._log_event('PREDICTIVE_REBALANCE', {'transfers': transfers, 'total': reclaimable})
        return {'performed': True, 'transfers': transfers, 'total': reclaimable, 'forecasts': forecasts}

    def update_opl(self, task_result):
        if not self.opl:
            return
        role_success, role_blocked = {}, {}
        for call in task_result.get('agent_calls', []):
            aid = call.get('agent_id', '')
            if aid in self.ledger:
                role = self.ledger[aid].role
                role_success[role] = call.get('success', False)
                role_blocked[role] = 'budget' in str(call.get('block_reason', '')).lower()
        self.opl.update(role_success, role_blocked)

    def update_credit(self, task_result):
        """Record credit repayment: an agent that borrowed and saw the task
        succeed gets a positive record; borrowed-but-failed is a default."""
        if not self.credit_scorer:
            return
        for acc in self.ledger.values():
            if acc.borrowed > 0:
                self.credit_scorer.record(acc.role, bool(task_result.get('success')))

    def _check_anomaly(self, agent_id, tokens):
        now = time.time()
        window = self._velocity_window[agent_id]
        window.append((now, tokens))
        window[:] = [(t, tok) for t, tok in window if now - t < 60]
        if len(window) < 3:
            return
        velocities = [tok for _, tok in window]
        mean_v, std_v = np.mean(velocities), np.std(velocities)
        if std_v > 0 and tokens > mean_v + self.anomaly_sigma * std_v:
            self.anomaly_log.append({'agent': agent_id, 'timestamp': now,
                                      'tokens': tokens, 'mean_v': mean_v, 'std_v': std_v})
            self._log_event('ANOMALY', {'agent': agent_id, 'tokens': tokens})

    def _compute_cost(self, tokens_in, tokens_out, model):
        p = PRICING.get(model, PRICING[PRIMARY_MODEL])
        return (tokens_in * p['input'] + tokens_out * p['output']) / 1e6

    def _log_event(self, event_type, data):
        self.trajectory.append({'type': event_type, 'time': time.time(), 'data': data, 'reserve': self.reserve})

    def get_summary(self):
        total_used = sum(a.used for a in self.ledger.values())
        total_cost = sum(a.total_cost for a in self.ledger.values())
        total_borrowed = sum(a.borrowed for a in self.ledger.values())
        return {
            'total_budget': self.total_budget, 'reserve': self.reserve,
            'total_used': total_used, 'total_cost_usd': round(total_cost, 4),
            'total_borrowed': total_borrowed, 'rebalance_count': self.rebalance_count,
            'borrow_count': self.borrow_count, 'anomaly_count': len(self.anomaly_log),
            'summarization_count': self.kv_tracker.summarization_count if self.kv_tracker else 0,
            'pacing_releases': self.pacing_releases,
            'agents': {k: {'quota': v.quota, 'used': v.used, 'borrowed': v.borrowed,
                          'utilization': round(v.utilization, 3), 'cost_usd': round(v.total_cost, 4)}
                     for k, v in self.ledger.items()}
        }

print('TokenBankV2 defined successfully.')

In [ ]:
from openai import OpenAI

class LLMClientV2:
    def __init__(self, api_key, base_url):
        if not MOCK_MODE:
            self.client = OpenAI(api_key=api_key, base_url=base_url)
        else:
            self.client = None
        self.call_count = 0

    def call(self, agent_id, bank, messages, model=PRIMARY_MODEL, max_tokens=MAX_TOKENS,
             temperature=0.7, action='inference', kv_cache=0):
        # KV-cache summarization BEFORE dispatch: compression shrinks the prompt,
        # which may bring the dispatch estimate back under quota. (Previously this
        # ran after can_dispatch, so an over-quota estimate blocked the call
        # before compression ever had a chance.)
        if bank.kv_tracker and agent_id in bank.ledger and bank.kv_tracker.should_summarize(bank.ledger[agent_id]):
            messages, _, kv_cache = bank.kv_tracker.summarize_context(messages, self)
            if VERBOSE:
                print(f'  [SUMMARIZE] {agent_id}: KV-cache compressed')

        prompt_text = '\n'.join([m.get('content', '') for m in messages])
        role = bank.ledger[agent_id].role if agent_id in bank.ledger else 'Executor'
        if MOCK_MODE:
            mean, std = MOCK_TOKEN_PROFILES.get(role, (10000, 3000))
            estimated_in = max(50, int(np.random.normal(mean * 0.5, std * 0.15)))
            estimated_out = max(1, int(np.random.normal(mean * 0.9, std * 0.3)))
            # Simulate KV-cache accumulation: the prompt context stays resident
            kv_cache = estimated_in
        else:
            estimated_in = len(ENCODER.encode(prompt_text))
            estimated_out = max_tokens
            # Multi-turn conversations keep prior context resident in KV cache
            if kv_cache == 0 and len(messages) > 2:
                kv_cache = estimated_in
        # Keep the dispatch estimate consistent with consume()'s accounting:
        # when KV tracking is on, consumption is charged as in + 0.3*kv + out.
        if bank.kv_tracker:
            pending_kv = bank.ledger[agent_id].kv_cache_tokens + kv_cache if agent_id in bank.ledger else kv_cache
            estimated_total = bank.kv_tracker.effective_tokens(estimated_in, pending_kv) + estimated_out
        else:
            estimated_total = estimated_in + estimated_out

        can_run, reason = bank.can_dispatch(agent_id, estimated_total)
        if not can_run:
            if VERBOSE:
                print(f'  [BLOCKED] {agent_id}: {reason}')
            return {'success': False, 'content': '', 'tokens_in': 0, 'tokens_out': 0,
                    'block_reason': reason, 'model': model, 'agent_id': agent_id}

        if MOCK_MODE:
            actual_out = estimated_out
            content = f'[MOCK_RESPONSE_{self.call_count}_{role}]'
            time.sleep(0.02)
        else:
            try:
                resp = self.client.chat.completions.create(model=model, messages=messages,
                                                            max_tokens=max_tokens, temperature=temperature,
                                                            extra_body=LLM_EXTRA_BODY or None)
                content = resp.choices[0].message.content or ''
                actual_out = resp.usage.completion_tokens if resp.usage else len(ENCODER.encode(content))
            except Exception as e:
                return {'success': False, 'content': '', 'error': str(e), 'agent_id': agent_id,
                        'tokens_in': 0, 'tokens_out': 0, 'model': model}

        actual_in = estimated_in if MOCK_MODE else len(ENCODER.encode(prompt_text))
        result = bank.consume(agent_id, actual_in, actual_out, model=model, action=action, kv_cache=kv_cache)
        if result['allowed']:
            bank.pacing_release_check(action)
        if not result['allowed']:
            content = content[:int(len(content) * result.get('actual_out', 0) / max(actual_out, 1))] if actual_out > 0 else ''
        self.call_count += 1
        if VERBOSE and result['allowed']:
            print(f'  [OK] {agent_id}: {actual_in}+{actual_out} tokens, cost=${result["cost_usd"]:.4f}')
        return {
            'success': result['allowed'], 'content': content, 'agent_id': agent_id,
            'tokens_in': result.get('actual_in', actual_in), 'tokens_out': result.get('actual_out', actual_out),
            'cost_usd': result.get('cost_usd', 0), 'model': result.get('model', model),
            'block_reason': result['reason'] if not result['allowed'] else None
        }

llm = LLMClientV2(OPENAI_API_KEY, OPENAI_BASE_URL)
print(f'LLMClientV2 ready. Mock={MOCK_MODE}')

In [ ]:
def _try_run_code(content: str, timeout: int = 10) -> Optional[str]:
    """If the Executor emitted a ```python block, run it in a subprocess
    (isolated, time-limited, dangerous operations rejected) and return its
    output. Returns None when there is no code block."""
    m = re.findall(r'```python\s*\n(.*?)```', content, re.DOTALL)
    if not m:
        return None
    code = m[0]
    if re.search(r'(import\s+os|import\s+sys|import\s+subprocess|__import__|open\s*\(|eval\s*\(|exec\s*\()', code):
        return '(blocked: restricted operation)'
    try:
        p = subprocess.run([sys.executable, '-c', code], capture_output=True, text=True, timeout=timeout)
        out = (p.stdout + p.stderr).strip()
        return out[:1000] if out else '(no output)'
    except Exception as e:
        return f'(execution failed: {e})'


def run_gaia_task_v2(task, bank, llm_client):
    results = {'task_id': task['id'], 'agent_calls': [], 'success': False, 'answer': ''}
    agents = {'planner': 'Planner', 'executor': 'Executor', 'reviewer': 'Reviewer'}
    bank.register_agents(agents)
    bank.allocate()

    r1 = llm_client.call('planner', bank, [
        {'role': 'system', 'content': 'You are a Planner. Break down the question into sub-tasks.'},
        {'role': 'user', 'content': f"Question: {task['question']}\nProvide a step-by-step plan."}
    ], action='plan')
    results['agent_calls'].append(r1)
    if not r1['success']:
        bank.update_opl(results)
        return results

    # Multi-round execute/review loop (up to 5 rounds) with a code-execution
    # tool: a ```python block in the Executor's reply is run and the result fed
    # back, so arithmetic/multi-hop questions get an extra competent shot.
    exec_messages = [
        {'role': 'system', 'content': 'You are an Executor. Answer concisely. '
         'For ANY computation, put Python code that prints the result in a ```python block; '
         'you will receive the output, then give the final answer.'},
        {'role': 'user', 'content': f"Question: {task['question']}\nPlan: {r1['content']}\nFinal answer."}
    ]
    max_rounds = 5
    r2, r3 = None, None
    for rnd in range(max_rounds):
        r2 = llm_client.call('executor', bank, exec_messages, action='execute')
        results['agent_calls'].append(r2)
        if not r2['success']:
            bank.update_opl(results)
            return results

        if not MOCK_MODE:
            tool_out = _try_run_code(r2['content'])
            if tool_out is not None:
                if VERBOSE:
                    print(f'  [TOOL] executor: code executed')
                exec_messages.append({'role': 'assistant', 'content': r2['content']})
                exec_messages.append({'role': 'user', 'content': f'[Computation result]\n{tool_out}\nNow give the final answer.'})
                r2 = llm_client.call('executor', bank, exec_messages, action='execute')
                results['agent_calls'].append(r2)
                if not r2['success']:
                    bank.update_opl(results)
                    return results

        r3 = llm_client.call('reviewer', bank, [
            {'role': 'system', 'content': 'You are a Reviewer. Validate the answer.'},
            {'role': 'user', 'content': f"Question: {task['question']}\nProposed: {r2['content']}\nCorrect? YES/NO"}
        ], action='review')
        results['agent_calls'].append(r3)
        bank.predictive_rebalance()

        if MOCK_MODE:
            approved = bool(np.random.random() < 0.6)
        else:
            approved = r3['success'] and 'yes' in r3['content'].lower()
        if approved or rnd == max_rounds - 1:
            break
        exec_messages.append({'role': 'assistant', 'content': r2['content']})
        exec_messages.append({'role': 'user', 'content': f"Reviewer feedback: {r3['content']}\nRevise and give the final answer."})

    if MOCK_MODE:
        # Mock 模式无法真实作答：按流水线完整性模拟答案正确性，
        # 有调用被预算拦截/截断时准确率显著降低
        blocked = any(not c['success'] for c in results['agent_calls'])
        acc = 0.30 if blocked else 0.85
        results['success'] = bool(np.random.random() < acc)
    else:
        ans = r2['content'].lower()
        gt = str(task['answer']).lower()
        results['success'] = gt in ans or any(w in ans for w in gt.split()[:2])
    results['answer'] = r2['content']
    bank.update_opl(results)
    return results


def run_swe_task_v2(task, bank, llm_client):
    results = {'task_id': task['id'], 'agent_calls': [], 'success': False, 'patch': ''}
    agents = {'planner': 'Planner', 'executor': 'Executor', 'debugger': 'Debugger', 'reviewer': 'Reviewer'}
    bank.register_agents(agents)
    bank.allocate()

    r1 = llm_client.call('planner', bank, [
        {'role': 'system', 'content': 'You are a code analyst.'},
        {'role': 'user', 'content': f"Issue: {task['issue']}\nExplain the bug briefly."}
    ], action='plan')
    results['agent_calls'].append(r1)
    if not r1['success']:
        return results

    r2 = llm_client.call('executor', bank, [
        {'role': 'system', 'content': 'You are a programmer. Output ONLY the fixed function.'},
        {'role': 'user', 'content': f"Fix: {task.get('code_before', '')}\nBug: {task['issue']}\nReturn complete fixed function."}
    ], action='execute')
    results['agent_calls'].append(r2)
    if not r2['success']:
        return results

    r3 = llm_client.call('debugger', bank, [
        {'role': 'system', 'content': 'You are a test engineer.'},
        {'role': 'user', 'content': f"Test: {task.get('test', '')}\nPatch: {r2['content']}\nPASS or FAIL?"}
    ], action='debug')
    results['agent_calls'].append(r3)

    r4 = llm_client.call('reviewer', bank, [
        {'role': 'system', 'content': 'You are a code reviewer.'},
        {'role': 'user', 'content': f"Issue: {task['issue']}\nPatch: {r2['content']}\nCorrect? YES/NO"}
    ], action='review')
    results['agent_calls'].append(r4)
    bank.predictive_rebalance()

    if MOCK_MODE:
        blocked = any(not c['success'] for c in results['agent_calls'])
        acc = 0.30 if blocked else 0.80
        results['success'] = bool(np.random.random() < acc)
    elif task.get('test'):
        # Fallback tasks carry unit tests: actually execute the extracted fix
        results['success'] = swe_eval_executable(task, r2['content'])
    else:
        # Real SWE-bench tasks: LLM-judge against the gold patch (reference-
        # based, not the participant's self-report). Judge runs outside bank
        # accounting on the cheap model.
        judge = llm_client.client.chat.completions.create(
            model=SUMMARIZE_MODEL, max_tokens=2 * MAX_TOKENS, temperature=0,
            extra_body=LLM_EXTRA_BODY or None,
            messages=[{'role': 'user', 'content':
                f"Gold fix:\n{task['gold_patch'][:2000]}\n\nProposed fix:\n{r2['content'][:2000]}\n\n"
                'Does the proposed fix implement essentially the same correction as '
                'the gold fix? Answer YES or NO.'}])
        verdict = (judge.choices[0].message.content or '').strip().lower()
        results['success'] = verdict.startswith('yes')
    results['patch'] = r2['content']
    bank.update_opl(results)
    return results


def _extract_code_block(text: str) -> str:
    m = re.findall(r'```(?:python)?\s*\n(.*?)```', text, re.DOTALL)
    return m[0] if m else text


def swe_eval_executable(task: Dict, patch_text: str) -> bool:
    """Execute the extracted fix against the task's unit tests (fallback tasks)."""
    code = _extract_code_block(patch_text)
    ns: Dict = {}
    try:
        exec(code, ns)
        exec(task['test'], ns)
        return True
    except Exception:
        return False


def run_financial_task_v2(task, bank, llm_client):
    results = {'task_id': task['id'], 'agent_calls': [], 'success': False, 'extracted': {}}
    agents = {'planner': 'Planner', 'executor': 'Executor', 'reviewer': 'Reviewer', 'debugger': 'Debugger'}
    bank.register_agents(agents)
    bank.allocate()

    r1 = llm_client.call('planner', bank, [
        {'role': 'system', 'content': 'You are a document analyst.'},
        {'role': 'user', 'content': f"Document excerpt: {task['document'][:500]}...\nList key fields to extract."}
    ], action='plan')
    results['agent_calls'].append(r1)
    if not r1['success']:
        return results

    fields_str = '\n'.join([f'- {k}' for k in task['fields'].keys()])
    r2 = llm_client.call('executor', bank, [
        {'role': 'system', 'content': 'You are an extraction engine. Output JSON only.'},
        {'role': 'user', 'content': f"Extract:\n{fields_str}\n\nDocument: {task['document']}\nReturn JSON."}
    ], action='execute')
    results['agent_calls'].append(r2)
    if not r2['success']:
        return results

    r3 = llm_client.call('reviewer', bank, [
        {'role': 'system', 'content': 'You are a data validator.'},
        {'role': 'user', 'content': f"Expected: {list(task['fields'].keys())}\nExtracted: {r2['content']}\nAll present? YES/NO"}
    ], action='review')
    results['agent_calls'].append(r3)

    r4 = llm_client.call('debugger', bank, [
        {'role': 'system', 'content': 'You are a data correction agent.'},
        {'role': 'user', 'content': f"Re-extract missing fields from: {task['document'][:300]}..."}
    ], action='debug')
    results['agent_calls'].append(r4)
    bank.predictive_rebalance()

    if MOCK_MODE:
        blocked = any(not c['success'] for c in results['agent_calls'])
        acc = 0.30 if blocked else 0.85
        results['success'] = bool(np.random.random() < acc)
        correct = int(len(task['fields']) * (0.9 if results['success'] else 0.5))
    else:
        extracted_text = (r2['content'] + ' ' + r4['content']).lower()
        correct = sum(1 for k, v in task['fields'].items() if v.lower() in extracted_text)
        results['success'] = correct >= len(task['fields']) * 0.75
    results['extracted'] = {'correct_fields': correct, 'total_fields': len(task['fields'])}
    bank.update_opl(results)
    return results

print('Task runners v2 defined.')

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

MAX_WORKERS = int(os.getenv('MAX_WORKERS', '16'))  # parallel tasks; V4 Pro limit is 500

def run_experiment_v2(tasks, task_runner, config, llm_client):
    budget = config.get('budget', 100000)
    policy = config.get('policy', 'hard')
    external_plan = config.get('external_plan', None)
    gamma = config.get('gamma', 0.8)
    enable_predictive = config.get('enable_predictive', True)
    enable_borrowing = config.get('enable_borrowing', True)
    enable_opl = config.get('enable_opl', True)
    enable_kv = config.get('enable_kv', True)
    theta = config.get('theta', 0.8)
    min_quota = config.get('min_quota', 5000)
    enable_credit_scoring = config.get('enable_credit_scoring', False)
    rogue_cutoff = config.get('rogue_cutoff', 0.10)
    credit_rate = config.get('credit_rate', 0.5)
    enable_pacing = config.get('enable_pacing', False)

    # One OPL per experiment run, shared across all task banks, so Thompson
    # Sampling accumulates over tasks and exits cold-start after 10 tasks
    shared_opl = (OnlinePriorityLearner(['Planner', 'Executor', 'Reviewer', 'Debugger'])
                  if enable_opl else None)
    # Optional shared credit scorer: sizes/prices credit lines per role from
    # the running history of credit-to-success conversion
    shared_credit_scorer = CreditScorer() if (enable_borrowing and enable_credit_scoring) else None

    def _run_one(task):
        # Each task gets its own bank, so tasks are independent and thread-safe
        bank = TokenBankV2(
            total_budget=budget, policy=policy, external_plan=external_plan, gamma=gamma,
            rebalance_threshold=theta, enable_predictive=enable_predictive,
            enable_borrowing=enable_borrowing, enable_opl=enable_opl, enable_kv=enable_kv,
            min_quota=min_quota, shared_opl=shared_opl,
            shared_credit_scorer=shared_credit_scorer,
            rogue_cutoff=rogue_cutoff, credit_rate=credit_rate,
            enable_pacing=enable_pacing
        )
        task_result = task_runner(task, bank, llm_client)
        bank.update_credit(task_result)
        task_result['bank_summary'] = bank.get_summary()
        task_result['config'] = config
        task_result['overrun'] = bank.reserve < 0 or any(a.used > a.quota + a.borrowed for a in bank.ledger.values())
        # Recovery: the task needed budget relief (credit or rebalancing) during
        # execution yet still succeeded — it would likely have failed under a
        # rigid quota but recovered through runtime governance.
        task_result['recovered'] = (task_result['success'] and
                                    (bank.borrow_count > 0 or bank.rebalance_count > 0))
        return task_result

    results = [None] * len(tasks)
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(_run_one, t): i for i, t in enumerate(tasks)}
        done = 0
        for fut in as_completed(futures):
            results[futures[fut]] = fut.result()
            done += 1
            if VERBOSE and done % 10 == 0:
                print(f'  Progress: {done}/{len(tasks)} tasks')

    total_cost = sum(r['bank_summary']['total_cost_usd'] for r in results)

    successes = [r['success'] for r in results]
    tokens_used = [r['bank_summary']['total_used'] for r in results]
    overruns = [r['overrun'] for r in results]
    recovered = [r['recovered'] for r in results]
    borrow_counts = [r['bank_summary']['borrow_count'] for r in results]

    return {
        'config': config, 'results': results, 'success_rate': np.mean(successes),
        'mean_tokens': np.mean(tokens_used), 'std_tokens': np.std(tokens_used),
        'overrun_rate': np.mean(overruns), 'recovery_rate': np.mean(recovered) if any(recovered) else 0.0,
        'total_cost_usd': total_cost, 'cost_per_success': total_cost / (sum(successes) or 1),
        'mean_borrows': np.mean(borrow_counts),
        'opl_tasks': shared_opl.task_count if shared_opl else 0,
        'opl_learned': shared_opl.get_priorities() if shared_opl else {},
        'credit_scores': shared_credit_scorer.summary() if shared_credit_scorer else {},
    }

print('Experiment orchestrator v2 defined.')

In [ ]:
# ==================== MAIN EXPERIMENT CONFIGURATIONS ====================
CONFIGS_V2 = [
    {'name': 'Unconstrained', 'budget': 200000, 'policy': 'hard', 'external_plan': None,
     'gamma': 0.0, 'enable_predictive': False, 'enable_borrowing': False, 'enable_opl': False, 'enable_kv': False, 'theta': 1.0},
    {'name': 'Static Quota', 'budget': 4000, 'min_quota': 800, 'policy': 'hard', 'external_plan': None,
     'gamma': 1.0, 'enable_predictive': False, 'enable_borrowing': False, 'enable_opl': False, 'enable_kv': False, 'theta': 1.0},
    {'name': 'TB-v1 (Hard)', 'budget': 4000, 'min_quota': 800, 'policy': 'hard', 'external_plan': None,
     'gamma': 0.0, 'enable_predictive': False, 'enable_borrowing': False, 'enable_opl': False, 'enable_kv': False, 'theta': 0.8},
    {'name': 'TB-v2-Full', 'budget': 4000, 'min_quota': 800, 'policy': 'hard', 'external_plan': None,
     'gamma': 0.8, 'enable_predictive': True, 'enable_borrowing': True, 'enable_opl': True, 'enable_kv': True,
     'theta': 0.8, 'credit_rate': 1.5},
    {'name': 'TB-v2-Grace', 'budget': 4000, 'min_quota': 800, 'policy': 'grace', 'external_plan': None,
     'gamma': 0.8, 'enable_predictive': True, 'enable_borrowing': True, 'enable_opl': True, 'enable_kv': True, 'theta': 0.8},
    {'name': 'TB-v2-Throttle', 'budget': 4000, 'min_quota': 800, 'policy': 'throttle', 'external_plan': None,
     'gamma': 0.8, 'enable_predictive': True, 'enable_borrowing': True, 'enable_opl': True, 'enable_kv': True, 'theta': 0.8},
    {'name': 'TB-v2-Scored', 'budget': 4000, 'min_quota': 800, 'policy': 'hard', 'external_plan': None,
     'gamma': 0.8, 'enable_predictive': True, 'enable_borrowing': True, 'enable_opl': True, 'enable_kv': True,
     'theta': 0.8, 'enable_credit_scoring': True, 'rogue_cutoff': 0.10},
]

# Select benchmark
BENCHMARK = os.getenv('BENCHMARK', 'GAIA')  # 'GAIA', 'SWE', 'FINANCIAL'
if BENCHMARK == 'GAIA':
    TASKS, RUNNER = GAIA_TASKS, run_gaia_task_v2
elif BENCHMARK == 'SWE':
    TASKS, RUNNER = SWE_TASKS, run_swe_task_v2
else:
    TASKS, RUNNER = FINANCIAL_TASKS, run_financial_task_v2

# Scale tight budgets per benchmark so budget pressure is comparable across
# task types (SWE prompts are much longer; Unconstrained stays at 200K)
BENCH_TIGHT = {'GAIA': (4000, 800), 'SWE': (10000, 2000), 'FINANCIAL': (8000, 1600)}
TIGHT_BUDGET, TIGHT_MIN_QUOTA = BENCH_TIGHT[BENCHMARK]
for cfg in CONFIGS_V2:
    if cfg.get('budget') != 200000:
        cfg['budget'] = TIGHT_BUDGET
        cfg['min_quota'] = TIGHT_MIN_QUOTA

# Multi-seed main experiment: SEEDS env var, e.g. SEEDS=42,43,44. Task set is
# fixed across seeds (shuffled once at load with the base SEED); seeds vary
# generation randomness. Results are pooled across seeds for statistics.
SEEDS = [int(s) for s in os.getenv('SEEDS', '42').split(',')]
print(f'Running {len(CONFIGS_V2)} configurations x {len(SEEDS)} seed(s) on {BENCHMARK} ({len(TASKS)} tasks)...')

raw_results_v2 = []
for seed in SEEDS:
    random.seed(seed)
    np.random.seed(seed)
    for cfg in CONFIGS_V2:
        print(f"\n{'='*60}")
        print(f'Running: {cfg["name"]} (seed={seed})')
        print(f"{'='*60}")
        result = run_experiment_v2(TASKS, RUNNER, cfg, llm)
        result['name'] = cfg['name']
        result['seed'] = seed
        raw_results_v2.append(result)
        print(f"  SR: {result['success_rate']:.1%} | Cost: ${result['total_cost_usd']:.4f}")

# Aggregate: pool task results across seeds; report per-seed SR spread
experiment_results_v2 = []
for cfg in CONFIGS_V2:
    runs = [r for r in raw_results_v2 if r['name'] == cfg['name']]
    pooled = [tr for r in runs for tr in r['results']]
    srs = [r['success_rate'] for r in runs]
    total_cost = sum(r['total_cost_usd'] for r in runs)
    agg = {
        'config': cfg, 'name': cfg['name'], 'results': pooled,
        'success_rate': np.mean([t['success'] for t in pooled]),
        'seed_sr_std': float(np.std(srs)) if len(srs) > 1 else 0.0,
        'mean_tokens': np.mean([t['bank_summary']['total_used'] for t in pooled]),
        'std_tokens': np.std([t['bank_summary']['total_used'] for t in pooled]),
        'overrun_rate': np.mean([t['overrun'] for t in pooled]),
        'recovery_rate': np.mean([t['recovered'] for t in pooled]),
        'total_cost_usd': total_cost,
        'cost_per_success': total_cost / (sum(t['success'] for t in pooled) or 1),
        'mean_borrows': np.mean([t['bank_summary']['borrow_count'] for t in pooled]),
        'opl_tasks': runs[-1].get('opl_tasks', 0), 'opl_learned': runs[-1].get('opl_learned', {}),
    }
    experiment_results_v2.append(agg)
    print(f"  {cfg['name']}: SR={agg['success_rate']:.1%}±{agg['seed_sr_std']:.1%} | "
          f"Tokens: {agg['mean_tokens']:.0f}±{agg['std_tokens']:.0f} | "
          f"RR: {agg['recovery_rate']:.1%} | Borrows: {agg['mean_borrows']:.1f} | "
          f"CPS: ${agg['cost_per_success']:.4f}")
    if agg['opl_tasks']:
        print(f"    OPL: {agg['opl_tasks']} tasks, priorities={agg['opl_learned']}")

print('\nAll experiments completed.')

In [ ]:
# ==================== ABLATION STUDY ====================
ABLATION_CONFIGS_V2 = [
    {'name': 'Full TB-v2', 'enable_predictive': True, 'enable_borrowing': True, 'enable_opl': True, 'enable_kv': True},
    {'name': 'w/o Predictive', 'enable_predictive': False, 'enable_borrowing': True, 'enable_opl': True, 'enable_kv': True},
    {'name': 'w/o Borrowing', 'enable_predictive': True, 'enable_borrowing': False, 'enable_opl': True, 'enable_kv': True},
    {'name': 'w/o OPL', 'enable_predictive': True, 'enable_borrowing': True, 'enable_opl': False, 'enable_kv': True},
    {'name': 'w/o KV', 'enable_predictive': True, 'enable_borrowing': True, 'enable_opl': True, 'enable_kv': False},
    {'name': 'Static Quota', 'enable_predictive': False, 'enable_borrowing': False, 'enable_opl': False, 'enable_kv': False},
]

base_cfg = {'budget': TIGHT_BUDGET, 'min_quota': TIGHT_MIN_QUOTA, 'policy': 'hard', 'gamma': 0.8, 'theta': 0.8, 'external_plan': None}
ablation_results_v2 = []

print('Running ablation study...')
for abl in ([] if SMOKE_N else ABLATION_CONFIGS_V2):
    cfg = {**base_cfg, **abl}
    result = run_experiment_v2(TASKS, RUNNER, cfg, llm)
    result['name'] = abl['name']
    ablation_results_v2.append(result)
    print(f"  {abl['name']}: SR={result['success_rate']:.1%}, Var={result['std_tokens']:.0f}, OR={result['overrun_rate']:.1%}, RR={result['recovery_rate']:.1%}")

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
names = [r['name'] for r in ablation_results_v2]
sr = [r['success_rate'] * 100 for r in ablation_results_v2]
colors = ['#22c55e', '#fbbf24', '#fb923c', '#f97316', '#a78bfa', '#ef4444']
bars = ax.bar(names, sr, color=colors, edgecolor='black', linewidth=1.2)
ax.set_ylabel('Success Rate (%)')
ax.set_title('Ablation Study: Component Contribution (v2)')
ax.set_ylim(80, 100)
for bar, val in zip(bars, sr):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{val:.1f}%',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig(f'ablation_v2_{BENCHMARK.lower()}{OUT_SUFFIX}.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ==================== SENSITIVITY: THETA ====================
THETA_VALUES = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
theta_results_v2 = []

print('Running sensitivity analysis on theta...')
for theta in ([] if SMOKE_N else THETA_VALUES):
    cfg = {'budget': TIGHT_BUDGET, 'min_quota': TIGHT_MIN_QUOTA, 'policy': 'hard', 'gamma': 0.8, 'theta': theta, 'external_plan': None,
           'enable_predictive': True, 'enable_borrowing': True, 'enable_opl': True, 'enable_kv': True}
    result = run_experiment_v2(TASKS, RUNNER, cfg, llm)
    result['theta'] = theta
    theta_results_v2.append(result)
    print(f'  theta={theta}: SR={result["success_rate"]:.1%}, OR={result["overrun_rate"]:.1%}, RR={result["recovery_rate"]:.1%}')

fig, ax1 = plt.subplots(figsize=(10, 5))
thetas = [r['theta'] for r in theta_results_v2]
sr_vals = [r['success_rate'] * 100 for r in theta_results_v2]
or_vals = [r['overrun_rate'] * 100 for r in theta_results_v2]
rr_vals = [r['recovery_rate'] * 100 for r in theta_results_v2]
ax2 = ax1.twinx()
l1 = ax1.plot(thetas, sr_vals, 'o-', color='#3b82f6', linewidth=2.5, markersize=10, label='Success Rate')
l2 = ax1.plot(thetas, rr_vals, '^-', color='#22c55e', linewidth=2.5, markersize=10, label='Recovery Rate')
l3 = ax2.plot(thetas, or_vals, 's--', color='#ef4444', linewidth=2.5, markersize=10, label='Overrun Rate')
ax1.set_xlabel('Rebalance Threshold theta')
ax1.set_ylabel('Rate (%)', color='#3b82f6')
ax2.set_ylabel('Overrun Rate (%)', color='#ef4444')
ax1.set_title('Sensitivity: Rebalance Threshold (v2)')
ax1.axvline(x=0.8, color='green', linestyle=':', linewidth=2, alpha=0.7)
ax1.set_ylim(80, 100)
ax2.set_ylim(-2, 15)
ax1.grid(True, alpha=0.3)
lines = l1 + l2 + l3
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='center left')
plt.tight_layout()
plt.savefig(f'sensitivity_theta_v2_{BENCHMARK.lower()}{OUT_SUFFIX}.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ==================== FULL VISUALIZATION ====================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f'Token Bank v2 Evaluation: {BENCHMARK}', fontsize=16, fontweight='bold')

names = [r['name'] for r in experiment_results_v2]
success_rates = [r['success_rate'] * 100 for r in experiment_results_v2]
mean_tokens = [r['mean_tokens'] / 1000 for r in experiment_results_v2]
std_tokens = [r['std_tokens'] / 1000 for r in experiment_results_v2]
overrun_rates = [r['overrun_rate'] * 100 for r in experiment_results_v2]
recovery_rates = [r['recovery_rate'] * 100 for r in experiment_results_v2]
costs = [r['total_cost_usd'] for r in experiment_results_v2]
colors = ['#ef4444', '#f97316', '#fbbf24', '#22c55e', '#3b82f6', '#8b5cf6', '#14b8a6']

# (a) Success Rate
ax = axes[0, 0]
bars = ax.bar(names, success_rates, color=colors, edgecolor='black')
ax.set_ylabel('Success Rate (%)')
ax.set_title('(a) Task Success Rate')
ax.set_ylim(80, 100)
ax.tick_params(axis='x', rotation=25)
for bar, val in zip(bars, success_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

# (b) Token Consumption
ax = axes[0, 1]
bars = ax.bar(names, mean_tokens, color=colors, edgecolor='black')
ax.set_ylabel('Tokens (K)')
ax.set_title('(b) Mean Token Consumption')
ax.tick_params(axis='x', rotation=25)
for bar, val in zip(bars, mean_tokens):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val:.1f}K', ha='center', va='bottom', fontsize=9, fontweight='bold')

# (c) Variance
ax = axes[0, 2]
bars = ax.bar(names, std_tokens, color=colors, edgecolor='black')
ax.set_ylabel('Std Dev (K)')
ax.set_title('(c) Consumption Variance')
ax.tick_params(axis='x', rotation=25)
for bar, val in zip(bars, std_tokens):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2, f'{val:.1f}K', ha='center', va='bottom', fontsize=9, fontweight='bold')

# (d) Overrun Rate
ax = axes[1, 0]
bars = ax.bar(names, overrun_rates, color=colors, edgecolor='black')
ax.set_ylabel('Overrun Rate (%)')
ax.set_title('(d) Budget Overrun Rate')
ax.tick_params(axis='x', rotation=25)
for bar, val in zip(bars, overrun_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

# (e) Recovery Rate (NEW METRIC)
ax = axes[1, 1]
bars = ax.bar(names, recovery_rates, color=colors, edgecolor='black')
ax.set_ylabel('Recovery Rate (%)')
ax.set_title('(e) Recovery After Budget Exhaustion')
ax.tick_params(axis='x', rotation=25)
for bar, val in zip(bars, recovery_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

# (f) Pareto Frontier
ax = axes[1, 2]
for i, (name, sr, cost) in enumerate(zip(names, success_rates, costs)):
    ax.scatter(cost, sr, s=300, c=colors[i], edgecolors='black', linewidth=2, zorder=5)
    ax.annotate(name, (cost, sr), textcoords='offset points', xytext=(8, 0), ha='left', fontsize=9, fontweight='bold')
ax.set_xlabel('Total Cost (USD)')
ax.set_ylabel('Success Rate (%)')
ax.set_title('(f) Pareto Frontier')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, max(costs) * 1.3)
plt.tight_layout()
plt.savefig(f'evaluation_v2_{BENCHMARK.lower()}{OUT_SUFFIX}.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ==================== STATISTICAL TESTS ====================
from scipy.stats import mannwhitneyu, chi2_contingency, levene

print('===== Statistical Significance Tests (v2) =====\n')

uc = [r for r in experiment_results_v2 if r['name'] == 'Unconstrained'][0]
tb2 = [r for r in experiment_results_v2 if r['name'] == 'TB-v2-Full'][0]

# 1. Token consumption
uc_tokens = [r['bank_summary']['total_used'] for r in uc['results']]
tb_tokens = [r['bank_summary']['total_used'] for r in tb2['results']]
stat, pval = mannwhitneyu(uc_tokens, tb_tokens, alternative='two-sided')
print(f'1. Token Consumption: Mann-Whitney U={stat:.2f}, p={pval:.4f}')

# 2. Success rate
uc_s = sum(r['success'] for r in uc['results'])
tb_s = sum(r['success'] for r in tb2['results'])
cont = [[uc_s, len(uc['results'])-uc_s], [tb_s, len(tb2['results'])-tb_s]]
if all(col > 0 for col in np.sum(cont, axis=0)):
    chi2, pval, _, _ = chi2_contingency(cont)
    print(f'2. Success Rate: Chi-square={chi2:.4f}, p={pval:.4f}')
else:
    print('2. Success Rate: Chi-square skipped (degenerate table: a row/column is all-zero)')

# 3. Variance
stat, pval = levene(uc_tokens, tb_tokens)
print(f'3. Variance (Levene): stat={stat:.4f}, p={pval:.4f}')

# 4. Recovery rate (unique to v2)
print(f'4. Recovery Rate: TB-v2={tb2["recovery_rate"]:.1%} vs Unconstrained={uc["recovery_rate"]:.1%}')

# 5. Borrow efficiency
borrows = [r['bank_summary']['borrow_count'] for r in tb2['results']]
print(f'5. Mean Borrows per Task: {np.mean(borrows):.2f}')

print('\nTests complete.')

In [ ]:
# ==================== PER-AGENT BREAKDOWN ====================
tb2_data = [r for r in experiment_results_v2 if r['name'] == 'TB-v2-Full'][0]
agent_stats = defaultdict(lambda: {'used': 0, 'quota': 0, 'borrowed': 0, 'calls': 0})

for task_result in tb2_data['results']:
    summary = task_result['bank_summary']
    for agent_id, data in summary['agents'].items():
        agent_stats[agent_id]['used'] += data['used']
        agent_stats[agent_id]['quota'] += data['quota']
        agent_stats[agent_id]['borrowed'] += data['borrowed']
        agent_stats[agent_id]['calls'] += len([c for c in task_result['agent_calls'] if c.get('success')])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
agents = list(agent_stats.keys())
used_vals = [agent_stats[a]['used'] / 1000 for a in agents]
quota_vals = [agent_stats[a]['quota'] / 1000 for a in agents]
borrow_vals = [agent_stats[a]['borrowed'] / 1000 for a in agents]
x = np.arange(len(agents))
width = 0.25
ax1.bar(x - width, quota_vals, width, label='Quota', color='#94a3b8', edgecolor='black')
ax1.bar(x, used_vals, width, label='Used', color='#3b82f6', edgecolor='black')
ax1.bar(x + width, borrow_vals, width, label='Borrowed', color='#f59e0b', edgecolor='black')
ax1.set_ylabel('Tokens (K)')
ax1.set_title('Per-Agent Quota vs. Used vs. Borrowed')
ax1.set_xticks(x)
ax1.set_xticklabels(agents)
ax1.legend()

util_rates = [agent_stats[a]['used'] / max(1, agent_stats[a]['quota']) * 100 for a in agents]
bars = ax2.bar(agents, util_rates, color=['#22c55e' if u < 70 else '#fbbf24' if u < 90 else '#ef4444' for u in util_rates], edgecolor='black')
ax2.set_ylabel('Utilization (%)')
ax2.set_title('Per-Agent Utilization Rate')
ax2.axhline(y=80, color='orange', linestyle='--', label='Rebalance Threshold')
ax2.axhline(y=100, color='red', linestyle='--', label='Quota Limit')
ax2.legend()
for bar, val in zip(bars, util_rates):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig(f'per_agent_v2_{BENCHMARK.lower()}{OUT_SUFFIX}.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ==================== EXPORT FINAL REPORT ====================
report = {
    'timestamp': datetime.now().isoformat(),
    'benchmark': BENCHMARK,
    'mock_mode': MOCK_MODE,
    'primary_model': PRIMARY_MODEL,
    'configurations': [
        {'name': r['name'], 'success_rate': round(r['success_rate'], 4),
         'seed_sr_std': round(r.get('seed_sr_std', 0.0), 4),
         'mean_tokens': round(r['mean_tokens'], 1), 'std_tokens': round(r['std_tokens'], 1),
         'overrun_rate': round(r['overrun_rate'], 4), 'recovery_rate': round(r['recovery_rate'], 4),
         'total_cost_usd': round(r['total_cost_usd'], 4), 'cost_per_success': round(r['cost_per_success'], 4),
         'mean_borrows': round(r['mean_borrows'], 2)}
        for r in experiment_results_v2
    ],
    'ablation': [
        {'name': r['name'], 'success_rate': round(r['success_rate'], 4),
         'std_tokens': round(r['std_tokens'], 1), 'overrun_rate': round(r['overrun_rate'], 4),
         'recovery_rate': round(r['recovery_rate'], 4)}
        for r in ablation_results_v2
    ],
    'sensitivity_theta': [
        {'theta': r['theta'], 'success_rate': round(r['success_rate'], 4),
         'overrun_rate': round(r['overrun_rate'], 4), 'recovery_rate': round(r['recovery_rate'], 4),
         'std_tokens': round(r['std_tokens'], 1)}
        for r in theta_results_v2
    ]
}

with open(f'experiment_report_v2_{BENCHMARK.lower()}{OUT_SUFFIX}.json', 'w') as f:
    json.dump(report, f, indent=2)

# Summary DataFrame
df_data = []
for r in experiment_results_v2:
    df_data.append({
        'Method': r['name'], 'Success Rate': f"{r['success_rate']:.1%}",
        'SR±seed': f"{r.get('seed_sr_std', 0):.1%}",
        'Mean Tokens': f"{r['mean_tokens']:.0f}", 'Std Tokens': f"{r['std_tokens']:.0f}",
        'Overrun': f"{r['overrun_rate']:.1%}", 'Recovery': f"{r['recovery_rate']:.1%}",
        'Cost ($)': f"{r['total_cost_usd']:.4f}", 'CPS ($)': f"{r['cost_per_success']:.4f}",
        'Borrows': f"{r['mean_borrows']:.1f}",
        'Rebalances': f"{np.mean([tr['bank_summary']['rebalance_count'] for tr in r['results']]):.2f}",
        'Summarizes': f"{np.mean([tr['bank_summary']['summarization_count'] for tr in r['results']]):.2f}"
    })
df = pd.DataFrame(df_data)
print(df.to_string(index=False))
df.to_csv(f'results_v2_{BENCHMARK.lower()}{OUT_SUFFIX}.csv', index=False)

print('\nExported files:')
print(f'  - experiment_report_v2_{BENCHMARK.lower()}{OUT_SUFFIX}.json')
print(f'  - results_v2_{BENCHMARK.lower()}{OUT_SUFFIX}.csv')
print(f'  - evaluation_v2_{BENCHMARK.lower()}{OUT_SUFFIX}.png')
print(f'  - ablation_v2_{BENCHMARK.lower()}{OUT_SUFFIX}.png')
print(f'  - sensitivity_theta_v2_{BENCHMARK.lower()}{OUT_SUFFIX}.png')
print(f'  - per_agent_v2_{BENCHMARK.lower()}{OUT_SUFFIX}.png')